In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("../../data/02-cleaned/cleaned_data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35335 entries, 0 to 35334
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   año                35335 non-null  int64  
 1   tkt_code           35335 non-null  int64  
 2   order_date         35335 non-null  str    
 3   order_code         35335 non-null  int64  
 4   start_time         35335 non-null  str    
 5   truck_code         35335 non-null  float64
 6   ship_plant_code    35335 non-null  int64  
 7   u_volumen          35335 non-null  float64
 8   typed_time         35335 non-null  str    
 9   at_plant_time      35335 non-null  str    
 10  u_cicle            35335 non-null  float64
 11  name               35335 non-null  str    
 12  nombredelproyecto  35335 non-null  str    
 13  ship_addr_line     35324 non-null  str    
dtypes: float64(3), int64(4), str(7)
memory usage: 3.8 MB


In [6]:
# keep relevant columns
columns_to_keep = ['order_date', 'truck_code', 'u_volumen', 'u_cicle']

df = df[columns_to_keep]
df.head()

,order_date,truck_code,u_volumen,u_cicle
0,2025-01-20 00:00:00,7039.0,1.5,50.0
1,2025-03-10 00:00:00,12863.0,7.0,77.0
2,2025-03-10 00:00:00,7044.0,7.0,100.0
3,2025-03-10 00:00:00,6302.0,2.0,83.0
4,2025-03-11 00:00:00,7044.0,3.5,60.0


In [7]:
# drop rows with missing values
df = df.dropna()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35335 entries, 0 to 35334
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_date  35335 non-null  str    
 1   truck_code  35335 non-null  float64
 2   u_volumen   35335 non-null  float64
 3   u_cicle     35335 non-null  float64
dtypes: float64(3), str(1)
memory usage: 1.1 MB


In [8]:
# create single date column
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed').dt.date
df.head()

,order_date,truck_code,u_volumen,u_cicle
0,2025-01-20,7039.0,1.5,50.0
1,2025-03-10,12863.0,7.0,77.0
2,2025-03-10,7044.0,7.0,100.0
3,2025-03-10,6302.0,2.0,83.0
4,2025-03-11,7044.0,3.5,60.0


In [9]:
# rename and reorder columns
df.rename(columns={'u_volumen': 'volume', 'u_cicle': 'cycle_time'}, inplace=True)
df = df[['order_date', 'truck_code', 'cycle_time', 'volume']]
df.head()

,order_date,truck_code,cycle_time,volume
0,2025-01-20,7039.0,50.0,1.5
1,2025-03-10,12863.0,77.0,7.0
2,2025-03-10,7044.0,100.0,7.0
3,2025-03-10,6302.0,83.0,2.0
4,2025-03-11,7044.0,60.0,3.5


In [10]:
# aggregate data by date and truck code
daily_truck_data = df.groupby(by=['order_date']).agg({
    'truck_code': 'nunique',
    'cycle_time': 'mean',
    'volume': 'sum'
}).reset_index()
daily_truck_data.head()

,order_date,truck_code,cycle_time,volume
0,2023-03-16,9,71.962963,115.5
1,2023-03-17,8,70.692308,80.5
2,2023-03-18,3,65.750000,13.5
3,2023-03-21,9,69.717949,148.0
4,2023-03-22,8,72.375000,143.0


In [11]:
# rename truck_code to truck_count
daily_truck_data = daily_truck_data.rename(columns={'truck_code': 'truck_count'})
daily_truck_data.head()

,order_date,truck_count,cycle_time,volume
0,2023-03-16,9,71.962963,115.5
1,2023-03-17,8,70.692308,80.5
2,2023-03-18,3,65.750000,13.5
3,2023-03-21,9,69.717949,148.0
4,2023-03-22,8,72.375000,143.0


In [12]:
# fill missing order_dates with 0s
daily_truck_data = daily_truck_data.set_index('order_date').asfreq('D', fill_value=0).reset_index()
daily_truck_data.head()

,order_date,truck_count,cycle_time,volume
0,2023-03-16,9,71.962963,115.5
1,2023-03-17,8,70.692308,80.5
2,2023-03-18,3,65.750000,13.5
3,2023-03-19,0,0.000000,0.0
4,2023-03-20,0,0.000000,0.0


In [13]:
# export to final csv
daily_truck_data.to_csv("../../data/03-final/daily_truck_data.csv", index=False)